# MM-CAD Inference: Tri-Modal Retrieval
Load the trained v4 model and run:
1. **Interactive text queries** — type a prompt, see top-10 retrieved CAD models
2. **Quantitative eval** — full Matryoshka retrieval matrix on train/val splits

In [ ]:
# Cell 1: Setup
import os, sys
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    !pip install -q sentence-transformers plyfile h5py
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# Cell: Extract images + sketches for visualization
import os, zipfile, time

if IS_COLAB:
    DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
    EXTRACT_DIR = '/content/mmcad_data'
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    # --- ISO images (images.zip) ---
    IMG_ROOT = None
    for candidate in [EXTRACT_DIR, os.path.join(EXTRACT_DIR, 'images')]:
        if os.path.isdir(os.path.join(candidate, 'abc_iso1_organized')):
            IMG_ROOT = candidate
            n = len(os.listdir(os.path.join(candidate, 'abc_iso1_organized')))
            print(f'ISO images already at: {candidate} ({n} files)')
            break
    if IMG_ROOT is None:
        zip_path = f'{DRIVE_DIR}/images.zip'
        if os.path.exists(zip_path):
            print(f'Extracting images.zip...')
            t0 = time.time()
            with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(EXTRACT_DIR)
            print(f'Done in {time.time()-t0:.0f}s')
            for candidate in [EXTRACT_DIR, os.path.join(EXTRACT_DIR, 'images')]:
                if os.path.isdir(os.path.join(candidate, 'abc_iso1_organized')):
                    IMG_ROOT = candidate; break
        else:
            print(f'images.zip not found at {zip_path}')
    print(f'IMG_ROOT: {IMG_ROOT}')

    # --- Sketch images (sketches.zip) ---
    # CSV sketch_iso1_path = 'sketches/xxxx_iso1.png_sketch.png'
    # So SKETCH_ROOT must be the PARENT of sketches/ directory
    SKETCH_ROOT = None
    for candidate in [EXTRACT_DIR, DRIVE_DIR]:
        sketch_dir = os.path.join(candidate, 'sketches')
        if os.path.isdir(sketch_dir) and len(os.listdir(sketch_dir)) > 100:
            SKETCH_ROOT = candidate  # parent of sketches/
            n = len(os.listdir(sketch_dir))
            print(f'Sketches already at: {sketch_dir} ({n} files)')
            break
    if SKETCH_ROOT is None:
        for archive_name in ['sketches.zip', 'sketch_images.zip']:
            archive = f'{DRIVE_DIR}/{archive_name}'
            if os.path.exists(archive):
                print(f'Extracting {archive_name}...')
                t0 = time.time()
                with zipfile.ZipFile(archive, 'r') as z: z.extractall(EXTRACT_DIR)
                print(f'Done in {time.time()-t0:.0f}s')
                # Check what directories were created
                for d in sorted(os.listdir(EXTRACT_DIR)):
                    full = os.path.join(EXTRACT_DIR, d)
                    if os.path.isdir(full): print(f'  Extracted dir: {d}/')
                SKETCH_ROOT = EXTRACT_DIR
                break
    if SKETCH_ROOT is None:
        if os.path.isdir(f'{DRIVE_DIR}/sketches'):
            SKETCH_ROOT = DRIVE_DIR
        else:
            print('WARNING: No sketch images found!')
            SKETCH_ROOT = EXTRACT_DIR
    print(f'SKETCH_ROOT: {SKETCH_ROOT}')

    # Verify sketch path resolution
    import pandas as pd
    _df_tmp = pd.read_csv(f'{DRIVE_DIR}/abc_dataset_clean.csv', nrows=5)
    _sample_path = str(_df_tmp[_df_tmp['has_sketch_iso1']==True].iloc[0]['sketch_iso1_path'])
    _full = os.path.join(SKETCH_ROOT, _sample_path)
    print(f'Sample sketch path: {_full}')
    print(f'  Exists: {os.path.exists(_full)}')
    del _df_tmp
else:
    IMG_ROOT = r'C:\Users\anush\Desktop\MMCAD'
    SKETCH_ROOT = r'C:\Users\anush\Desktop\MMCAD'
    print(f'IMG_ROOT: {IMG_ROOT}')
    print(f'SKETCH_ROOT: {SKETCH_ROOT}')


In [ ]:
# Cell 2: Imports
import os, math, time, gc, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import h5py
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from plyfile import PlyData

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Config
class Config:
    if IS_COLAB:
        DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
        CSV_PATH = f'{DRIVE_DIR}/abc_dataset_clean.csv'
        ANCHOR_DIR = f'{DRIVE_DIR}/sketch_anchors'
        V4_CKPT = f'{DRIVE_DIR}/baseline_trimodal_v4/latest.pth'
        SKETCH_CKPT = f'{DRIVE_DIR}/sketch_encoder_v1/best.pth'
        PLY_H5 = f'{DRIVE_DIR}/abc_point_clouds.h5'
    else:
        DATA_ROOT = r'C:\Users\anush\Desktop\MMCAD'
        CSV_PATH = os.path.join(DATA_ROOT, 'abc_dataset_clean.csv')
        ANCHOR_DIR = os.path.join(DATA_ROOT, 'sketch_anchors')
        V4_CKPT = os.path.join(DATA_ROOT, 'baseline_trimodal_v4', 'latest.pth')
        SKETCH_CKPT = os.path.join(DATA_ROOT, 'sketch_encoder_v1', 'best.pth')
        PLY_H5 = os.path.join(DATA_ROOT, 'abc_point_clouds.h5')

    D_SHARED = 768
    MATRYOSHKA_DIMS = [128, 256, 512, 768]
    TEXT_MODEL_GEMMA = 'google/embeddinggemma-300m'
    TEXT_MAX_LENGTH = 512
    VIT_MODEL = 'google/vit-base-patch16-224'
    DGCNN_K = 20
    DGCNN_LATENT = 1024
    NUM_POINTS = 2048
    # V4 model config (needed for Baseline class)
    FACE_DIM = 16
    D_BREP_MODEL = 512
    N_BREP_LAYERS = 6
    N_HEADS = 8
    MAX_FACES = 192
    MAX_EDGES = 512
    K_VALUES = [1, 5, 10, 25]
    SEED = 42

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'K values: {config.K_VALUES}')



In [ ]:
# Cell 4: Model definitions
# ALL model classes needed so v4 checkpoint loads with strict=True.

# --- Text Encoders ---
class EmbeddingGemmaEncoder(nn.Module):
    def __init__(self, model_id='google/embeddinggemma-300m'):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer(model_id, trust_remote_code=True)
        self._pipeline = nn.ModuleList(list(st))
        self.tokenizer = st.tokenizer
        print(f'EmbeddingGemma: {sum(p.numel() for p in self.parameters())/1e6:.1f}M params')
    def forward(self, input_ids, attention_mask):
        features = {'input_ids': input_ids, 'attention_mask': attention_mask}
        for m in self._pipeline: features = m(features)
        return features['sentence_embedding']

class BertTextEncoder(nn.Module):
    def __init__(self, model_name='bert-base-uncased', d_out=768):
        super().__init__()
        from transformers import BertModel, BertTokenizer
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)
        self.projection = nn.Linear(768, d_out)
    def forward(self, input_ids, attention_mask):
        return self.projection(self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :])

# --- ViT Sketch Encoder ---
from transformers import ViTModel
class ViTSketchEncoder(nn.Module):
    def __init__(self, model_name='google/vit-base-patch16-224', d_out=768):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        h = self.vit.config.hidden_size
        self.projection = nn.Sequential(
            nn.Linear(h, d_out), nn.LayerNorm(d_out),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(d_out, d_out))
    def forward(self, pixel_values):
        return self.projection(self.vit(pixel_values=pixel_values).last_hidden_state[:, 0])

# --- DGCNN ---
def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    return (-xx - inner - xx.transpose(2, 1)).topk(k=k, dim=-1)[1]
def get_graph_feature(x, k=20):
    bs, d, n = x.size(); idx = knn(x, k)
    idx_base = torch.arange(0, bs, device=x.device).view(-1, 1, 1) * n
    idx = (idx + idx_base).view(-1); x = x.transpose(2, 1).contiguous()
    feat = x.view(bs * n, -1)[idx].view(bs, n, k, d)
    x = x.view(bs, n, 1, d).repeat(1, 1, k, 1)
    return torch.cat((feat - x, x), dim=3).permute(0, 3, 1, 2).contiguous()
class DGCNNEncoder(nn.Module):
    def __init__(self, latent_size=1024, k=20):
        super().__init__()
        self.k = k
        self.bn1, self.bn2 = nn.BatchNorm2d(64), nn.BatchNorm2d(64)
        self.bn3, self.bn4 = nn.BatchNorm2d(128), nn.BatchNorm2d(256)
        self.bn5, self.bn6, self.bn7 = nn.BatchNorm1d(latent_size), nn.BatchNorm1d(512), nn.BatchNorm1d(latent_size)
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, 1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, 1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, latent_size, 1, bias=False), self.bn5, nn.LeakyReLU(0.2))
        self.linear1 = nn.Linear(latent_size * 2, 512, bias=False)
        self.linear2 = nn.Linear(512, latent_size)
        self.dp = nn.Dropout(0.3)
    def forward(self, x):
        x1 = self.conv1(get_graph_feature(x, self.k)).max(dim=-1)[0]
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(dim=-1)[0]
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(dim=-1)[0]
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(dim=-1)[0]
        x = self.conv5(torch.cat((x1, x2, x3, x4), dim=1))
        x = torch.cat((x.max(2)[0], x.mean(2)), 1)
        x = self.dp(F.leaky_relu(self.bn6(self.linear1(x)), 0.2))
        return self.dp(self.bn7(self.linear2(x)))
class DGCNNWithProjection(nn.Module):
    def __init__(self, d_out=768, k=20, latent_size=1024):
        super().__init__()
        self.dgcnn = DGCNNEncoder(latent_size=latent_size, k=k)
        self.projection = nn.Sequential(
            nn.Linear(latent_size, d_out), nn.LayerNorm(d_out),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(d_out, d_out))
    def forward(self, x): return self.projection(self.dgcnn(x))

# --- BRepFormer ---
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.eps = eps; self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.scale * x / torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
class SwiGLUFFN(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.w1 = nn.Linear(d_in, d_hidden, bias=True)
        self.w2 = nn.Linear(d_hidden, d_in, bias=True)
        self.w3 = nn.Linear(d_in, d_hidden, bias=True)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))
class BRepFormerLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, ffn_mult=4, dropout=0.1):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = SwiGLUFFN(d_model, d_model * ffn_mult)
        self.attn_drop = nn.Dropout(dropout)
    def forward(self, x, attn_bias=None, mask=None):
        B, N, D = x.shape; h = self.norm1(x)
        q = self.q_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if attn_bias is not None: attn = attn + attn_bias
        if mask is not None: attn = attn.masked_fill(mask[:, None, None, :] == 0, float('-inf'))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        x = x + self.out_proj(out); x = x + self.ffn(self.norm2(x)); return x
class BRepFormerEncoder(nn.Module):
    def __init__(self, d_face_in=16, d_out=768, d_model=512, n_heads=8, n_layers=6, max_faces=192):
        super().__init__()
        self.d_model, self.max_faces = d_model, max_faces
        self.face_proj = nn.Linear(d_face_in, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.topo_bias_proj = nn.Sequential(
            nn.Linear(3, d_model), RMSNorm(d_model), nn.ReLU(), nn.Linear(d_model, n_heads))
        self.layers = nn.ModuleList([BRepFormerLayer(d_model, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.output_proj = nn.Sequential(nn.Linear(d_model, d_out), nn.GELU(), nn.Linear(d_out, d_out))
    def forward(self, face_features, face_centroids, edge_to_faces, face_mask, face_normals):
        pass  # Not needed for inference — we use pre-computed BRep anchors

# --- Baseline (v4 wrapper — needed to load checkpoint) ---
class Baseline(nn.Module):
    def __init__(self, text_encoder, config):
        super().__init__()
        self.text_encoder = text_encoder
        self.brep_encoder = BRepFormerEncoder(
            d_face_in=config.FACE_DIM, d_out=config.D_SHARED,
            d_model=config.D_BREP_MODEL, n_heads=config.N_HEADS,
            n_layers=config.N_BREP_LAYERS, max_faces=config.MAX_FACES)
        self.pc_encoder = DGCNNWithProjection(
            d_out=config.D_SHARED, k=config.DGCNN_K, latent_size=config.DGCNN_LATENT)
        self.log_tau = nn.Parameter(torch.log(torch.tensor(0.07)))
        self.matryoshka_dims = config.MATRYOSHKA_DIMS

print('All model classes defined (including Baseline for v4 checkpoint loading).')


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Cell 5: Load encoders + pre-compute/load ALL embeddings (cached to Drive)
# Text, sketch, PC embeddings are computed ONCE and saved.
# Subsequent runs just load the .pt files (instant).

import psutil
from torchvision import transforms
from PIL import Image

EMB_DIR = config.ANCHOR_DIR  # same dir as brep_anchors
BREP_PATH = os.path.join(EMB_DIR, 'brep_anchors.pt')
TEXT_PATH = os.path.join(EMB_DIR, 'text_embeddings.pt')
SKETCH_PATH = os.path.join(EMB_DIR, 'sketch_embeddings.pt')
PC_PATH = os.path.join(EMB_DIR, 'pc_embeddings.pt')
SPLIT_PATH = os.path.join(EMB_DIR, 'split_info.pt')

# --- Load what already exists ---
brep_embs = torch.load(BREP_PATH, map_location='cpu', weights_only=True)
print(f'BRep embeddings: {len(brep_embs)}')

split_info = torch.load(SPLIT_PATH, map_location='cpu', weights_only=True)
train_uids = split_info['train_uids']
val_uids = split_info['val_uids']
all_uids_list = train_uids + val_uids
print(f'Split: {len(train_uids)} train, {len(val_uids)} val, {len(all_uids_list)} total')

# --- Text embeddings: compute if missing ---
# Delete broken cached text embeddings if they exist (from previous strict=False bug)
_force_recompute_text = False
if os.path.exists(TEXT_PATH):
    _tmp = torch.load(TEXT_PATH, map_location='cpu', weights_only=True)
    # Quick sanity check: text embeddings should have high cosine sim to brep for same UID
    _test_uids = [u for u in list(_tmp.keys())[:100] if u in brep_embs]
    if len(_test_uids) > 10:
        _zt = F.normalize(torch.stack([_tmp[u] for u in _test_uids]), dim=-1)
        _zb = F.normalize(torch.stack([brep_embs[u] for u in _test_uids]), dim=-1)
        _diag_sim = (_zt * _zb).sum(dim=-1).mean().item()
        print(f'Text embedding sanity check: avg diagonal cosine sim = {_diag_sim:.4f}')
        if _diag_sim < 0.05:
            print('WARNING: Text embeddings appear broken (near-zero alignment with BRep).')
            print('Deleting cached file and recomputing...')
            os.remove(TEXT_PATH)
            _force_recompute_text = True
        else:
            text_embs = _tmp
            print(f'Text embeddings loaded: {len(text_embs)}')
    else:
        text_embs = _tmp
        print(f'Text embeddings loaded: {len(text_embs)} (could not sanity check)')
    del _tmp

if not os.path.exists(TEXT_PATH):
    print('Computing text embeddings via full Baseline model (one-time)...')
    print('Loading full v4 model (guarantees correct text encoder weights)...')
    try:
        _text_enc = EmbeddingGemmaEncoder(config.TEXT_MODEL_GEMMA)
        _tokenizer = _text_enc.tokenizer
    except Exception as e:
        print(f'EmbeddingGemma failed: {e}, using BERT')
        _text_enc = BertTextEncoder(d_out=config.D_SHARED)
        _tokenizer = _text_enc.tokenizer
    _model = Baseline(_text_enc, config).to(device)
    v4 = torch.load(config.V4_CKPT, map_location=device, weights_only=False)
    _model.load_state_dict(v4['model_state_dict'])  # strict=True (default)!
    _model.eval()
    print(f'Loaded v4 checkpoint from epoch {v4["epoch"]+1} (strict=True)')
    del v4
    # Use the correctly-loaded text encoder
    _text_encoder = _model.text_encoder
    df_tmp = pd.read_csv(config.CSV_PATH)
    df_tmp['uid_str'] = df_tmp['uid'].astype(str).str.strip()
    uid_to_text = {}
    for _, r in df_tmp.iterrows():
        u = str(r['uid']).strip()
        t = str(r['title']) if pd.notna(r['title']) else ''
        d = str(r['description']) if pd.notna(r['description']) else ''
        uid_to_text[u] = f'title: none | text: {t}. {d}' if d else f'title: none | text: {t}'
    text_embs = {}
    BS = 256
    with torch.no_grad():
        for start in tqdm(range(0, len(all_uids_list), BS), desc='Text'):
            batch = all_uids_list[start:start+BS]
            texts = [uid_to_text.get(u, 'title: none | text: ') for u in batch]
            tok = _tokenizer(texts, padding=True, truncation=True, max_length=config.TEXT_MAX_LENGTH, return_tensors='pt')
            z = _text_encoder(tok['input_ids'].to(device), tok['attention_mask'].to(device))
            for i, u in enumerate(batch): text_embs[u] = z[i].cpu()
            if start % 5000 == 0: torch.cuda.empty_cache()
    # Sanity check before saving
    _test_uids = [u for u in list(text_embs.keys())[:100] if u in brep_embs]
    _zt = F.normalize(torch.stack([text_embs[u] for u in _test_uids]), dim=-1)
    _zb = F.normalize(torch.stack([brep_embs[u] for u in _test_uids]), dim=-1)
    _diag_sim = (_zt * _zb).sum(dim=-1).mean().item()
    print(f'Sanity check: avg diagonal cosine sim = {_diag_sim:.4f}')
    if _diag_sim < 0.05:
        print('ERROR: Text embeddings still broken! Check v4 checkpoint.')
    else:
        torch.save(text_embs, TEXT_PATH)
        print(f'Text embeddings saved: {len(text_embs)}')
    del _model, _text_encoder, _tokenizer, uid_to_text, df_tmp
    torch.cuda.empty_cache(); gc.collect()

# --- Sketch embeddings: compute if missing ---
SKETCH_TRANSFORM = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
sketch_model = None
if os.path.exists(SKETCH_PATH):
    sketch_embs = torch.load(SKETCH_PATH, map_location='cpu', weights_only=True)
    print(f'Sketch embeddings loaded: {len(sketch_embs)}')
    if os.path.exists(config.SKETCH_CKPT):
        sketch_model = ViTSketchEncoder(config.VIT_MODEL, d_out=config.D_SHARED).to(device)
        sk_ckpt = torch.load(config.SKETCH_CKPT, map_location=device, weights_only=False)
        sketch_model.load_state_dict(sk_ckpt['model_state_dict']); sketch_model.eval(); del sk_ckpt
else:
    print('Computing sketch embeddings (one-time)...')
    sketch_model = ViTSketchEncoder(config.VIT_MODEL, d_out=config.D_SHARED).to(device)
    if os.path.exists(config.SKETCH_CKPT):
        sk_ckpt = torch.load(config.SKETCH_CKPT, map_location=device, weights_only=False)
        sketch_model.load_state_dict(sk_ckpt['model_state_dict']); sketch_model.eval(); del sk_ckpt
    else: print('WARNING: sketch checkpoint not found')
    df_tmp = pd.read_csv(config.CSV_PATH)
    df_tmp['uid_str'] = df_tmp['uid'].astype(str).str.strip()
    sketch_embs = {}
    batch_imgs, batch_uids = [], []
    with torch.no_grad():
        for _, r in tqdm(df_tmp.iterrows(), total=len(df_tmp), desc='Sketch'):
            u = str(r['uid']).strip()
            if r.get('has_sketch_iso1', False) != True: continue
            sp = os.path.join(SKETCH_ROOT, str(r['sketch_iso1_path']))
            if not os.path.exists(sp): continue
            try:
                img = Image.open(sp).convert('RGB')
                batch_imgs.append(SKETCH_TRANSFORM(img)); batch_uids.append(u)
            except: continue
            if len(batch_imgs) >= 512:
                z = sketch_model(torch.stack(batch_imgs).to(device)).cpu()
                for j, uid in enumerate(batch_uids): sketch_embs[uid] = z[j]
                batch_imgs, batch_uids = [], []
                torch.cuda.empty_cache()
        if batch_imgs:
            z = sketch_model(torch.stack(batch_imgs).to(device)).cpu()
            for j, uid in enumerate(batch_uids): sketch_embs[uid] = z[j]
    torch.save(sketch_embs, SKETCH_PATH)
    print(f'Sketch embeddings saved: {len(sketch_embs)}')
    del df_tmp; torch.cuda.empty_cache(); gc.collect()

# --- PC embeddings: load if available ---
pc_embs = None
if os.path.exists(PC_PATH):
    pc_embs = torch.load(PC_PATH, map_location='cpu', weights_only=True)
    print(f'PC embeddings loaded: {len(pc_embs)}')
else:
    print(f'PC embeddings not found at {PC_PATH} (pc2brep will be skipped)')
    print('To generate: run the PC extraction cell below, or pre-compute separately.')

# --- Load text encoder for custom queries ---
# (Reload after potential cleanup above)
# Load text encoder for custom queries (via full Baseline for correct weights)
try:
    _te = EmbeddingGemmaEncoder(config.TEXT_MODEL_GEMMA)
    tokenizer = _te.tokenizer
except:
    _te = BertTextEncoder(d_out=config.D_SHARED)
    tokenizer = _te.tokenizer
_baseline = Baseline(_te, config).to(device)
v4 = torch.load(config.V4_CKPT, map_location=device, weights_only=False)
_baseline.load_state_dict(v4['model_state_dict'])  # strict=True!
_baseline.eval()
text_encoder = _baseline.text_encoder  # reference to correctly loaded encoder
del v4
# Don't delete _baseline yet — text_encoder is a reference into it
print(f'Text encoder loaded for custom queries (via Baseline, strict=True)')

print(f'\nRAM: {psutil.virtual_memory().used/1e9:.1f} GB')



In [ ]:
# Cell 6: Load CSV metadata

df = pd.read_csv(config.CSV_PATH)
df['uid_str'] = df['uid'].astype(str).str.strip()
print(f'CSV: {len(df)} rows')

# Build uid -> row index for fast metadata lookup
uid_to_row = dict(zip(df['uid_str'], range(len(df))))


In [ ]:
# Cell 7: Build gallery tensors for all splits and modalities

def build_gallery(uid_list, name):
    valid = [u for u in uid_list if u in brep_embs]
    g = {
        'brep': F.normalize(torch.stack([brep_embs[u] for u in valid]), dim=-1),
        'text': F.normalize(torch.stack([text_embs.get(u, torch.zeros(config.D_SHARED)) for u in valid]), dim=-1),
        'uids': valid,
    }
    # Sketch (sparse — not all samples have sketches)
    sk_vecs = [sketch_embs.get(u, torch.zeros(config.D_SHARED)) for u in valid]
    g['sketch'] = torch.stack(sk_vecs)
    g['sketch_mask'] = torch.tensor([u in sketch_embs for u in valid])
    # PC (optional)
    if pc_embs is not None:
        pc_vecs = [pc_embs.get(u, torch.zeros(config.D_SHARED)) for u in valid]
        g['pc'] = F.normalize(torch.stack(pc_vecs), dim=-1)
        g['pc_mask'] = torch.tensor([u in pc_embs for u in valid])
    # UID -> index for fast lookup
    g['uid2idx'] = {u: i for i, u in enumerate(valid)}
    print(f'  {name}: {len(valid)} samples, sketch={int(g["sketch_mask"].sum())}'
          + (f', pc={int(g["pc_mask"].sum())}' if 'pc_mask' in g else ''))
    return g

print('Building galleries...')
G = {}
G['val'] = build_gallery(val_uids, 'val')
G['train'] = build_gallery(train_uids, 'train')
G['all'] = build_gallery(all_uids_list, 'all')

# Helper: get df for a split
df_splits = {
    'val': df[df['uid_str'].isin(set(G['val']['uids']))].reset_index(drop=True),
    'train': df[df['uid_str'].isin(set(G['train']['uids']))].reset_index(drop=True),
    'all': df[df['uid_str'].isin(set(G['all']['uids']))].reset_index(drop=True),
}


In [ ]:
# Cell 8: Query embedding + fusion functions

@torch.no_grad()
def embed_text(query):
    prompted = f'title: none | text: {query}'
    tok = tokenizer(prompted, padding=True, truncation=True,
                    max_length=config.TEXT_MAX_LENGTH, return_tensors='pt')
    z = text_encoder(tok['input_ids'].to(device), tok['attention_mask'].to(device))
    return F.normalize(z, dim=-1).cpu()

@torch.no_grad()
def embed_sketch(img_path):
    img = Image.open(img_path).convert('RGB')
    z = sketch_model(SKETCH_TRANSFORM(img).unsqueeze(0).to(device))
    return F.normalize(z, dim=-1).cpu()

def fuse_embeddings(z_text, z_sketch, method='average', weight=0.5):
    if method == 'average': return F.normalize(z_text + z_sketch, dim=-1)
    elif method == 'weighted': return F.normalize(weight * z_text + (1-weight) * z_sketch, dim=-1)
    elif method == 'score_fusion': return None
    elif method == 'max_pool': return F.normalize(torch.max(z_text, z_sketch), dim=-1)
    else: raise ValueError(f'Unknown: {method}')

print('Query functions ready.')


In [ ]:
# Cell 9: Summary
for k, g in G.items():
    print(f'{k}: {len(g["uids"])} samples')


In [ ]:
# Cell 10: Retrieval function

def retrieve(query, top_k=10, split='all', query_type='text',
             fusion='average', fusion_weight=0.5,
             sketch_path=None, text_query=None, query_uid=None):
    g = G[split]
    z_gallery = g['brep']

    if query_type == 'text':
        z_q = embed_text(query)
        sims = (z_q @ z_gallery.T).squeeze(0)
    elif query_type == 'sketch':
        z_q = embed_sketch(query)
        sims = (z_q @ z_gallery.T).squeeze(0)
    elif query_type == 'sketch+text':
        s_path = query if sketch_path is None else sketch_path
        t_q = text_query or ''
        z_t, z_s = embed_text(t_q), embed_sketch(s_path)
        if fusion == 'score_fusion':
            sims = fusion_weight * (z_t @ z_gallery.T).squeeze(0) + (1-fusion_weight) * (z_s @ z_gallery.T).squeeze(0)
        else:
            z_q = fuse_embeddings(z_t, z_s, fusion, fusion_weight)
            sims = (z_q @ z_gallery.T).squeeze(0)
    else: raise ValueError(query_type)

    topk = sims.topk(top_k)
    results = []
    for rank, (idx, sim) in enumerate(zip(topk.indices.tolist(), topk.values.tolist())):
        uid = g['uids'][idx]
        rows = df[df['uid_str'] == uid]
        if len(rows) == 0: continue
        r = rows.iloc[0]
        results.append({'rank': rank+1, 'uid': uid, 'similarity': f'{sim:.4f}',
            'title': r.get('title',''), 'description': str(r.get('description',''))[:80],
            'is_match': uid == query_uid if query_uid else False})
    return pd.DataFrame(results)

print('retrieve() ready.')


In [ ]:
# Cell 11: Visual retrieval with green highlights for correct matches
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def _show_iso1(ax, uid):
    if IMG_ROOT is None: return False
    iso1 = df[df['uid_str'] == uid]['iso1_path'].values
    if len(iso1) == 0: return False
    p = os.path.join(IMG_ROOT, str(iso1[0]))
    if not os.path.exists(p): return False
    try: ax.imshow(mpimg.imread(p)); ax.axis('off'); return True
    except: return False

def visual_retrieve(query, top_k=10, split='all', query_type='text',
                    fusion='average', fusion_weight=0.5,
                    sketch_path=None, text_query=None, query_uid=None):
    results_df = retrieve(query, top_k=top_k, split=split, query_type=query_type,
        fusion=fusion, fusion_weight=fusion_weight,
        sketch_path=sketch_path, text_query=text_query, query_uid=query_uid)
    n = len(results_df)
    if n == 0: print('No results.'); return

    show_q = query_type in ('sketch','sketch+text')
    q_img = query if sketch_path is None else sketch_path
    if show_q and not os.path.exists(str(q_img)): show_q = False
    total = n + (1 if show_q else 0)
    cols = min(6, total); rows = (total + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.8*rows))
    axes = np.array(axes).reshape(-1)

    if query_type == 'text': ql = str(query)[:55]
    elif query_type == 'sketch': ql = os.path.basename(str(query))
    else: ql = f'{os.path.basename(str(q_img))} + "{(text_query or "")[:30]}" [{fusion}]'
    fig.suptitle(f'{query_type}: {ql}', fontsize=11, y=1.02)

    pi = 0
    if show_q:
        ax = axes[pi]
        try: ax.imshow(mpimg.imread(str(q_img)))
        except: ax.text(0.5,0.5,'Query',ha='center',va='center',transform=ax.transAxes)
        ax.set_title('QUERY', fontsize=10, fontweight='bold', color='blue')
        ax.axis('off')
        for s in ax.spines.values(): s.set_visible(True); s.set_color('blue'); s.set_linewidth(3)
        pi += 1

    for i, (_, row) in enumerate(results_df.iterrows()):
        ax = axes[pi]; uid = str(row['uid']); sim = row['similarity']; title = str(row['title'])[:30]
        is_match = row.get('is_match', False)
        if not _show_iso1(ax, uid):
            ax.text(0.5,0.5,f'UID:{uid}',ha='center',va='center',transform=ax.transAxes,fontsize=8)
            ax.axis('off')
        color = 'green' if is_match else 'black'
        ax.set_title(f'#{i+1} sim={sim}\n{title}', fontsize=8, color=color, fontweight='bold' if is_match else 'normal')
        if is_match:
            for s in ax.spines.values(): s.set_visible(True); s.set_color('green'); s.set_linewidth(3)
        pi += 1
    for j in range(pi, len(axes)): axes[j].axis('off')
    plt.tight_layout(); plt.show()

print('visual_retrieve() ready. Green border+title = correct match.')


In [ ]:
# Cell 12: Quantitative eval (INSTANT — all pre-computed, chunked similarity)

def compute_recall_chunked(z_q, z_g, k_values, chunk=2048):
    'Compute R@k with chunked sim to avoid OOM on large galleries.'
    N = z_q.shape[0]; max_k = max(k_values)
    correct = {k: 0 for k in k_values}
    z_g_gpu = z_g.to(device)
    for start in range(0, N, chunk):
        end = min(start + chunk, N)
        sim = z_q[start:end].to(device) @ z_g_gpu.T
        topk = sim.topk(min(max_k, sim.shape[1]), dim=1).indices
        gt = torch.arange(start, end, device=device).unsqueeze(1)
        for k in k_values:
            correct[k] += (topk[:, :k] == gt).any(dim=1).sum().item()
    del z_g_gpu; torch.cuda.empty_cache()
    return {k: 100.0 * correct[k] / N for k in k_values}


def full_eval(split='val'):
    'Instant eval using pre-computed embeddings. No encoding needed.'
    g = G[split]; N = len(g['uids'])
    print(f'\n=== {split.upper()} eval ({N} samples) ===')
    zb = g['brep']
    results = {}

    for dim in config.MATRYOSHKA_DIMS:
        zb_d = F.normalize(zb[:, :dim], dim=-1)

        # text2brep
        zt_d = F.normalize(g['text'][:, :dim], dim=-1)
        for k, v in compute_recall_chunked(zt_d, zb_d, config.K_VALUES).items():
            results[f'text2brep_R@{k}_{dim}d'] = v

        # pc2brep
        if 'pc' in g and 'pc_mask' in g and g['pc_mask'].sum() > 0:
            m = g['pc_mask']
            zp_d = F.normalize(g['pc'][m, :dim], dim=-1)
            zb_m = zb_d[m] if m.sum() == N else zb_d  # if all have PC, same gallery
            # For pc2brep, we need full gallery but masked ground truth
            pc_N = int(m.sum())
            z_g_gpu = zb_d.to(device)
            gt_indices = torch.arange(N)[m]
            correct_pc = {k: 0 for k in config.K_VALUES}
            max_k = max(config.K_VALUES)
            for start in range(0, pc_N, 2048):
                end = min(start + 2048, pc_N)
                sim = zp_d[start:end].to(device) @ z_g_gpu.T
                topk = sim.topk(min(max_k, sim.shape[1]), dim=1).indices
                gt_chunk = gt_indices[start:end].to(device).unsqueeze(1)
                for k in config.K_VALUES:
                    correct_pc[k] += (topk[:, :k] == gt_chunk).any(dim=1).sum().item()
            del z_g_gpu; torch.cuda.empty_cache()
            for k in config.K_VALUES:
                results[f'pc2brep_R@{k}_{dim}d'] = 100.0 * correct_pc[k] / pc_N

        # sketch2brep
        m = g['sketch_mask']
        if m.sum() > 0:
            zs_d = F.normalize(g['sketch'][m, :dim], dim=-1)
            sk_N = int(m.sum())
            z_g_gpu = zb_d.to(device)
            gt_indices = torch.arange(N)[m]
            correct_sk = {k: 0 for k in config.K_VALUES}
            max_k = max(config.K_VALUES)
            for start in range(0, sk_N, 2048):
                end = min(start + 2048, sk_N)
                sim = zs_d[start:end].to(device) @ z_g_gpu.T
                topk = sim.topk(min(max_k, sim.shape[1]), dim=1).indices
                gt_chunk = gt_indices[start:end].to(device).unsqueeze(1)
                for k in config.K_VALUES:
                    correct_sk[k] += (topk[:, :k] == gt_chunk).any(dim=1).sum().item()
            del z_g_gpu; torch.cuda.empty_cache()
            for k in config.K_VALUES:
                results[f'sketch2brep_R@{k}_{dim}d'] = 100.0 * correct_sk[k] / sk_N

            # Fusion methods (sketch+text -> brep)
            zt_m = F.normalize(g['text'][m, :dim], dim=-1)
            for method in ['average', 'score_fusion']:
                if method == 'score_fusion':
                    # Late fusion: combine sim scores
                    z_g_gpu = zb_d.to(device)
                    correct_f = {k: 0 for k in config.K_VALUES}
                    for start in range(0, sk_N, 2048):
                        end = min(start + 2048, sk_N)
                        sim_t = zt_m[start:end].to(device) @ z_g_gpu.T
                        sim_s = zs_d[start:end].to(device) @ z_g_gpu.T
                        sim_f = 0.5 * sim_t + 0.5 * sim_s
                        topk = sim_f.topk(min(max_k, sim_f.shape[1]), dim=1).indices
                        gt_chunk = gt_indices[start:end].to(device).unsqueeze(1)
                        for k in config.K_VALUES:
                            correct_f[k] += (topk[:, :k] == gt_chunk).any(dim=1).sum().item()
                    del z_g_gpu; torch.cuda.empty_cache()
                    for k in config.K_VALUES:
                        results[f'{method}_R@{k}_{dim}d'] = 100.0 * correct_f[k] / sk_N
                else:  # average
                    zf = F.normalize(zt_m + zs_d, dim=-1)
                    z_g_gpu = zb_d.to(device)
                    correct_f = {k: 0 for k in config.K_VALUES}
                    for start in range(0, sk_N, 2048):
                        end = min(start + 2048, sk_N)
                        sim = zf[start:end].to(device) @ z_g_gpu.T
                        topk = sim.topk(min(max_k, sim.shape[1]), dim=1).indices
                        gt_chunk = gt_indices[start:end].to(device).unsqueeze(1)
                        for k in config.K_VALUES:
                            correct_f[k] += (topk[:, :k] == gt_chunk).any(dim=1).sum().item()
                    del z_g_gpu; torch.cuda.empty_cache()
                    for k in config.K_VALUES:
                        results[f'{method}_R@{k}_{dim}d'] = 100.0 * correct_f[k] / sk_N

    # Print table
    dirs = ['text2brep']
    if 'pc' in g and g.get('pc_mask', torch.tensor([])).sum() > 0: dirs.append('pc2brep')
    if g['sketch_mask'].sum() > 0: dirs.extend(['sketch2brep', 'average', 'score_fusion'])
    for metric in config.K_VALUES:
        mstr = f'R@{metric}'
        print(f'\n{split.upper()} {mstr:>6}', end='')
        for dim in config.MATRYOSHKA_DIMS: print(f'{dim}d'.rjust(10), end='')
        print()
        print('-' * (16 + 10 * len(config.MATRYOSHKA_DIMS)))
        for d in dirs:
            print(f'{d:>16}', end='')
            for dim in config.MATRYOSHKA_DIMS:
                v = results.get(f'{d}_{mstr}_{dim}d', 0)
                print(f'{v:>9.2f}%', end='')
            print()
    return results

print('full_eval() ready. Instant — no encoding needed.')


In [ ]:
# Cell 13: Val set evaluation
val_results = full_eval('val')


In [ ]:
# Cell 14: Full dataset evaluation
all_results = full_eval('all')


In [ ]:
# Cell 15: Text-to-BRep retrieval — random samples from dataset
# Re-run this cell for different random samples!

SPLIT = 'all'     # 'val' or 'all'
N_SAMPLES = 3    # number of random queries
TOP_K = 10

g = G[SPLIT]; dfs = df_splits[SPLIT]
sample_idx = np.random.choice(len(g['uids']), size=N_SAMPLES, replace=False)

for idx in sample_idx:
    uid = g['uids'][idx]
    rows = df[df['uid_str'] == uid]
    if len(rows) == 0: continue
    r = rows.iloc[0]
    title = str(r.get('title', '')); desc = str(r.get('description', ''))
    text = f'{title}. {desc}' if desc and desc != 'nan' else title
    print(f'\nUID: {uid} | Text: {text[:80]}')
    visual_retrieve(text, top_k=TOP_K, split=SPLIT, query_type='text', query_uid=uid)


In [ ]:
# Cell 16: Sketch-to-BRep retrieval — random samples
# Re-run for different random samples!

SPLIT = 'all'
N_SAMPLES = 3
TOP_K = 10

if sketch_model is not None:
    g = G[SPLIT]
    sketch_uids = [g['uids'][i] for i in range(len(g['uids'])) if g['sketch_mask'][i]]
    if len(sketch_uids) > 0:
        sample_uids = np.random.choice(sketch_uids, size=min(N_SAMPLES, len(sketch_uids)), replace=False)
        for uid in sample_uids:
            rows = df[df['uid_str'] == uid]
            if len(rows) == 0: continue
            r = rows.iloc[0]
            sp = os.path.join(SKETCH_ROOT, str(r['sketch_iso1_path']))
            if not os.path.exists(sp): continue
            print(f'\nUID: {uid} | {r.get("title","")}')
            visual_retrieve(sp, top_k=TOP_K, split=SPLIT, query_type='sketch', query_uid=uid)
    else: print('No sketches in gallery.')
else: print('Sketch encoder not loaded.')


In [ ]:
# Cell 17: Sketch+Text fusion — compare average vs score_fusion
# Re-run for different random samples!

SPLIT = 'all'
N_SAMPLES = 2
TOP_K = 10

if sketch_model is not None:
    g = G[SPLIT]
    sketch_uids = [g['uids'][i] for i in range(len(g['uids'])) if g['sketch_mask'][i]]
    sample_uids = np.random.choice(sketch_uids, size=min(N_SAMPLES, len(sketch_uids)), replace=False)
    for uid in sample_uids:
        rows = df[df['uid_str'] == uid]
        if len(rows) == 0: continue
        r = rows.iloc[0]
        sp = os.path.join(SKETCH_ROOT, str(r['sketch_iso1_path']))
        if not os.path.exists(sp): continue
        title = str(r.get('title','')); desc = str(r.get('description',''))
        text = f'{title}. {desc}' if desc and desc != 'nan' else title
        print(f'\nUID: {uid}')
        print(f'Text: {text[:80]}')
        for method in ['average', 'score_fusion']:
            visual_retrieve(sp, text_query=text, top_k=TOP_K, split=SPLIT,
                            query_type='sketch+text', fusion=method, query_uid=uid)
else: print('Sketch encoder not loaded.')


In [ ]:
# Cell 18: Custom input — edit and run!

SPLIT = 'all'  # 'val' or 'all'

# === Text query ===
# visual_retrieve('cylindrical gear with 20 teeth', top_k=10, split=SPLIT, query_type='text')

# === Sketch query (provide path to a .png) ===
# visual_retrieve('/path/to/sketch.png', top_k=10, split=SPLIT, query_type='sketch')

# === Sketch + Text fusion ===
# visual_retrieve('/path/to/sketch.png', text_query='gear with teeth',
#                 top_k=10, split=SPLIT, query_type='sketch+text', fusion='average')
